## 1、数据的存储

1. 从TXT文档中加载数据，向量化后存储到Chroma数据库

In [3]:
from importlib.metadata import metadata

from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
# 创建阿里向量模型
from langchain_community.embeddings import DashScopeEmbeddings
import os
import dotenv

# 步骤一：创建加载器
loader = TextLoader(
    file_path='./asset/load/09-ai1.txt',
    encoding='utf-8'
)
docs = loader.load()

# 步骤2: 创建文本分割器，并拆分文档
text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)
spitter_docs = text_splitter.split_documents(docs)
print(len(spitter_docs))

# 步骤3: 创建模型
dotenv.load_dotenv(dotenv_path='../chapter02-model IO/.env')
os.environ["DASHSCOPE_API_KEY"] = os.getenv("OPENAI_API_KEY2")
embedding = DashScopeEmbeddings(model="text-embedding-v3")

# 步骤4: 创建向量数据库
"""
 persist_directory存储到本地文件，如果我们使用的from_documents()中没有显式的
 指明存储位置的话，则将当前的数据存储在内存中，并缓存起来，如果需要指明具体的存储位置，
 需要设置参数 persist_directory的值
"""

# db1=Chroma.from_documents(spitter_docs, embedding)
# 需要明确，向量数据库中，不仅存储了数据（或文档）的向量，而且还存储了数据本身
db2 = Chroma.from_documents(spitter_docs, embedding, persist_directory="./asset/vector_db")





3


In [5]:
# 向量的检索
query = "人工智能的核心技术有哪些？"

docs = db2.similarity_search(query)
# print(docs)

print(docs[0].page_content)




3. 人工智能的核心技术
3.1 机器学习
机器学习是人工智能的核心技术之一，通过算法使计算机从数据中学习并做出决策。常见的机器学习算法包括监督学习、无监督学习和强化学习。监督学习通过标记数据进行训练，无监督学习则从未标记数据中寻找模式，强化学习则通过与环境交互来优化决策。
3.2 深度学习
深度学习是机器学习的一个子领域，通过多层神经网络进行特征提取和模式识别。深度学习在图像识别、自然语言处理、语音识别等领域取得了显著成果。常见的深度学习模型包括卷积神经网络（CNN）、循环神经网络（RNN）和长短期记忆网络（LSTM）。
3.3 自然语言处理
自然语言处理（NLP）是人工智能的一个重要分支，致力于使计算机能够理解和生成人类语言。NLP技术广泛应用于机器翻译、情感分析、文本分类等领域。近年来，基于深度学习的NLP模型（如BERT、GPT）在语言理解任务中取得了突破性进展。
3.4 计算机视觉
计算机视觉是人工智能的另一个重要分支，致力于使计算机能够理解和处理图像和视频。计算机视觉技术广泛应用于图像识别、目标检测、人脸识别等领域。深度学习模型（如CNN）在计算机视觉任务中取得了显著成果。

4. 人工智能的应用领域
4.1 医疗健康
人工智能在医疗健康领域的应用包括疾病诊断、药物研发、个性化医疗等。通过分析医学影像和患者数据，人工智能可以帮助医生更准确地诊断疾病，提高治疗效果。
4.2 金融
人工智能在金融领域的应用包括风险评估、欺诈检测、算法交易等。通过分析市场数据和交易记录，人工智能可以帮助金融机构做出更明智的决策，提高运营效率。
4.3 教育
人工智能在教育领域的应用包括个性化学习、智能辅导、自动评分等。通过分析学生的学习数据，人工智能可以为学生提供个性化的学习建议，提高学习效果。
4.4 交通
人工智能在交通领域的应用包括自动驾驶、交通管理、智能导航等。通过分析交通数据和路况信息，人工智能可以帮助优化交通流量，提高交通安全。


In [11]:
# 举例2:操作csv文档，创建向量数据库
from langchain_community.document_loaders import CSVLoader
from langchain_text_splitters import CharacterTextSplitter
import os
import dotenv

loader = loader = CSVLoader(
    file_path='./asset/load/03-load.csv',
    encoding='utf-8'
)
docs = loader.load()
# print(len(docs)) 4

# 文本拆分
text_splitter = CharacterTextSplitter(
    chunk_size=500,
)
spitter_docs = text_splitter.split_documents(docs)

# 步骤3: 创建模型
dotenv.load_dotenv(dotenv_path='../chapter02-model IO/.env')
os.environ["DASHSCOPE_API_KEY"] = os.getenv("OPENAI_API_KEY2")
embedding = DashScopeEmbeddings(model="text-embedding-v3")
db = Chroma.from_documents(spitter_docs, embedding, persist_directory="./asset/vector1_db")

In [23]:
# 向量的检索
query = "title标题有哪些？"

docs = db.similarity_search(query)
# print(docs)

for doc in docs:
    print(doc.page_content)

id: 3
title: Web Development
content: HTML, CSS and JavaScript are core web technologies.
author: Mike Johnson
id: 4
title: Artificial Intelligence
content: AI is transforming many industries.
author: Sarah Williams
id: 1
title: Introduction to Python
content: Python is a popular programming language.
author: John Doe
id: 2
title: Data Science Basics
content: Data science involves statistics and machine learning.
author: Jane Smith


## 2、向量的检索

In [7]:
# 1.导入相关依赖
from langchain_core.documents import Document
# 创建阿里向量模型
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_community.vectorstores import Chroma
import os
import dotenv

# 2.自定义文档
raw_documents = [
    Document(
        page_content="黑洞是时空曲率极大以至于光都无法逃逸的天体，由大质量恒星坍缩形成。其边界称为事件视界，中心为奇点。2019年，事件视界望远镜发布了首张黑洞（M87*）照片。",
        metadata={"source": "黑洞", "type": "天文"}
    ),
    Document(
        page_content="太阳系由太阳和围绕它运行的行星、矮行星、小行星、彗星等天体组成。八大行星按距离太阳由近及远依次为：水星、金星、地球、火星、木星、土星、天王星、海王星。木星是最大的行星，土星拥有显著的行星环。",
        metadata={"source": "太阳系", "type": "天文"}
    ),
    Document(
        page_content="喜马拉雅山脉是世界海拔最高的山脉，位于青藏高原南缘，全长约2400公里。主峰珠穆朗玛峰海拔8848.86米，为地球最高峰。山脉形成于印度板块与欧亚板块的碰撞挤压。",
        metadata={"source": "喜马拉雅山脉", "type": "地理"}
    ),
    Document(
        page_content="秦始皇（公元前259年－公元前210年），嬴姓赵氏，名政，是中国历史上首位完成华夏大一统的皇帝。他建立了中央集权制度，推行书同文、车同轨、统一度量衡，并修筑万里长城。",
        metadata={"source": "秦始皇", "type": "历史"}
    ),
    Document(
        page_content="Python是一种解释型、高级、通用的编程语言，由吉多·范罗苏姆于1989年底发明。其设计哲学强调代码可读性和简洁语法，支持面向对象、函数式等多种编程范式。广泛应用于数据分析、人工智能、Web开发等领域。",
        metadata={"source": "Python", "type": "计算机"}
    ),
    Document(
        page_content="DNA（脱氧核糖核酸）是生物体内携带遗传信息的分子，呈双螺旋结构。由核苷酸组成，每个核苷酸包含磷酸、脱氧核糖和四种碱基（A、T、C、G）。DNA通过复制和转录指导蛋白质合成，决定生物性状。",
        metadata={"source": "DNA", "type": "医学/生物"}
    ),
    Document(
        page_content="审计是最崩溃的行业，无尽的加班",
        metadata={"source": "审计", "type": "岗位"}
    ),
    Document(
        page_content="《蒙娜丽莎》是文艺复兴时期达·芬奇创作的油画，现藏于卢浮宫博物馆。画作以神秘的微笑和渐隐法（晕涂法）著称，背景山水朦胧。该作品是西方艺术史上最著名的肖像画之一。",
        metadata={"source": "蒙娜丽莎", "type": "艺术"}
    ),
    Document(
        page_content="足球是世界上最受欢迎的运动之一，每四年举办一次的国际足联世界杯是最高荣誉赛事。标准比赛每队11人，主要用脚控球，以进球多者为胜。巴西队是夺得世界杯冠军次数最多的国家（5次）。",
        metadata={"source": "足球", "type": "体育"}
    )]
# 3.创建嵌入模型
dotenv.load_dotenv(dotenv_path='../chapter02-model IO/.env')
os.environ["DASHSCOPE_API_KEY"] = os.getenv("OPENAI_API_KEY2")
embedding = DashScopeEmbeddings(model="text-embedding-v3")

# 4.创建向量数据库
db = Chroma.from_documents(documents=raw_documents, embedding=embedding, persist_directory="./asset/vector2_db")






In [3]:
# 检索：相似性检索
query = '天文'
docs = db.similarity_search(query, k=3) # k=3表示返回最相似的3个文档
print(f"查询：{query}的结果")
for i, doc in enumerate(docs, 1):
    print(f"\n结果{i}:")
    print(f"内容：{doc.page_content}")
    print(f"元数据：{doc.metadata}")



查询：天文的结果

结果1:
内容：太阳系由太阳和围绕它运行的行星、矮行星、小行星、彗星等天体组成。八大行星按距离太阳由近及远依次为：水星、金星、地球、火星、木星、土星、天王星、海王星。木星是最大的行星，土星拥有显著的行星环。
元数据：{'type': '天文', 'source': '太阳系'}

结果2:
内容：喜马拉雅山脉是世界海拔最高的山脉，位于青藏高原南缘，全长约2400公里。主峰珠穆朗玛峰海拔8848.86米，为地球最高峰。山脉形成于印度板块与欧亚板块的碰撞挤压。
元数据：{'type': '地理', 'source': '喜马拉雅山脉'}

结果3:
内容：Python是一种解释型、高级、通用的编程语言，由吉多·范罗苏姆于1989年底发明。其设计哲学强调代码可读性和简洁语法，支持面向对象、函数式等多种编程范式。广泛应用于数据分析、人工智能、Web开发等领域。
元数据：{'type': '计算机', 'source': 'Python'}


In [4]:
# 支持直接对问题向量查询
embedding_vector = embedding.embed_query(query)
docs = db.similarity_search_by_vector(embedding_vector, k=3)
print(f"查询：{query}的结果")
for i, doc in enumerate(docs, 1):
    print(f"\n结果{i}:")
    print(f"内容：{doc.page_content}")
    print(f"元数据：{doc.metadata}")




查询：天文的结果

结果1:
内容：太阳系由太阳和围绕它运行的行星、矮行星、小行星、彗星等天体组成。八大行星按距离太阳由近及远依次为：水星、金星、地球、火星、木星、土星、天王星、海王星。木星是最大的行星，土星拥有显著的行星环。
元数据：{'type': '天文', 'source': '太阳系'}

结果2:
内容：喜马拉雅山脉是世界海拔最高的山脉，位于青藏高原南缘，全长约2400公里。主峰珠穆朗玛峰海拔8848.86米，为地球最高峰。山脉形成于印度板块与欧亚板块的碰撞挤压。
元数据：{'source': '喜马拉雅山脉', 'type': '地理'}

结果3:
内容：Python是一种解释型、高级、通用的编程语言，由吉多·范罗苏姆于1989年底发明。其设计哲学强调代码可读性和简洁语法，支持面向对象、函数式等多种编程范式。广泛应用于数据分析、人工智能、Web开发等领域。
元数据：{'source': 'Python', 'type': '计算机'}


In [5]:
# 相似性检索，支持过滤元数据
query='国际足联比赛'
docs=db.similarity_search(
    query=query,
    k=3,
    # 过滤掉其他内容
    filter={"source":"足球"}
)
for i,doc in enumerate(docs, 1):
    print(f"\n结果{i}:")
    print(f"内容：{doc.page_content}")
    print(f"元数据：{doc.metadata}")




结果1:
内容：足球是世界上最受欢迎的运动之一，每四年举办一次的国际足联世界杯是最高荣誉赛事。标准比赛每队11人，主要用脚控球，以进球多者为胜。巴西队是夺得世界杯冠军次数最多的国家（5次）。
元数据：{'type': '体育', 'source': '足球'}


In [8]:
# 通过L2距离分数进行搜索(similarity_search_with_score)
docs=db.similarity_search_with_score("什么行业最崩溃")

# 分数值越小，检索到的文档越和问题相似。分数取值范围：[0，正无穷]
for doc,score in docs:
    print(f"[L2距离得分={score:.3f}]{doc.page_content}[{doc.metadata}]")



[L2距离得分=0.466]审计是最崩溃的行业，无尽的加班[{'type': '岗位', 'source': '审计'}]
[L2距离得分=1.292]黑洞是时空曲率极大以至于光都无法逃逸的天体，由大质量恒星坍缩形成。其边界称为事件视界，中心为奇点。2019年，事件视界望远镜发布了首张黑洞（M87*）照片。[{'source': '黑洞', 'type': '天文'}]
[L2距离得分=1.360]喜马拉雅山脉是世界海拔最高的山脉，位于青藏高原南缘，全长约2400公里。主峰珠穆朗玛峰海拔8848.86米，为地球最高峰。山脉形成于印度板块与欧亚板块的碰撞挤压。[{'source': '喜马拉雅山脉', 'type': '地理'}]
[L2距离得分=1.360]喜马拉雅山脉是世界海拔最高的山脉，位于青藏高原南缘，全长约2400公里。主峰珠穆朗玛峰海拔8848.86米，为地球最高峰。山脉形成于印度板块与欧亚板块的碰撞挤压。[{'source': '喜马拉雅山脉', 'type': '地理'}]


In [14]:
# 通过余弦相似度分数进行搜索 取值范围[-1,1] 越接近1越相似
docs=db.similarity_search_with_relevance_scores("什么行业最崩溃")

# print(docs)
for doc,score in docs:
    # print(doc,'----',score)
    print(f"[余弦相似度得分={score:.3f}]{doc.page_content}[{doc.metadata}]")


[余弦相似度得分=0.670]审计是最崩溃的行业，无尽的加班[{'source': '审计', 'type': '岗位'}]
[余弦相似度得分=0.087]黑洞是时空曲率极大以至于光都无法逃逸的天体，由大质量恒星坍缩形成。其边界称为事件视界，中心为奇点。2019年，事件视界望远镜发布了首张黑洞（M87*）照片。[{'type': '天文', 'source': '黑洞'}]
[余弦相似度得分=0.038]喜马拉雅山脉是世界海拔最高的山脉，位于青藏高原南缘，全长约2400公里。主峰珠穆朗玛峰海拔8848.86米，为地球最高峰。山脉形成于印度板块与欧亚板块的碰撞挤压。[{'source': '喜马拉雅山脉', 'type': '地理'}]
[余弦相似度得分=0.038]喜马拉雅山脉是世界海拔最高的山脉，位于青藏高原南缘，全长约2400公里。主峰珠穆朗玛峰海拔8848.86米，为地球最高峰。山脉形成于印度板块与欧亚板块的碰撞挤压。[{'type': '地理', 'source': '喜马拉雅山脉'}]


In [15]:
# 最大边际相关性
docs=db.max_marginal_relevance_search(
    "什么行业最崩溃",
    # 范围0～1,其中0对应最大多样性，1对应最小多样性。默认值为0.5
    lambda_mult=0.8 #侧重相似性
)
for i, doc in enumerate(docs, 1):
    print(f"\n结果{i+1}")
    print(f"内容:{doc.page_content}")
    print(f"标签：{'. '.join(f'{k}={v}'for k,v in doc.metadata.items())}")



结果2
内容:审计是最崩溃的行业，无尽的加班
标签：source=审计. type=岗位

结果3
内容:黑洞是时空曲率极大以至于光都无法逃逸的天体，由大质量恒星坍缩形成。其边界称为事件视界，中心为奇点。2019年，事件视界望远镜发布了首张黑洞（M87*）照片。
标签：source=黑洞. type=天文

结果4
内容:喜马拉雅山脉是世界海拔最高的山脉，位于青藏高原南缘，全长约2400公里。主峰珠穆朗玛峰海拔8848.86米，为地球最高峰。山脉形成于印度板块与欧亚板块的碰撞挤压。
标签：type=地理. source=喜马拉雅山脉

结果5
内容:Python是一种解释型、高级、通用的编程语言，由吉多·范罗苏姆于1989年底发明。其设计哲学强调代码可读性和简洁语法，支持面向对象、函数式等多种编程范式。广泛应用于数据分析、人工智能、Web开发等领域。
标签：source=Python. type=计算机
